In [2]:
# =============================================================================
# Zelle 01 – Setup & Imports
# =============================================================================
# Ziel: Umgebung fuer die synthetische Datengenerierung (Modell A) vorbereiten.
# Reproduzierbarkeit ueber festen Seed sichergestellt (wie in Vorprojekten).
# Hinweis: Alle Firmen-/Produktbezuege in diesem Projekt sind fiktiv
# (Extruder GmbH).
# =============================================================================

import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN

# Reproduzierbarkeit
SEED = 42
rng = np.random.default_rng(SEED)

# Store44-Stil aktivieren (gilt ab jetzt fuer alle Plots in diesem Notebook)
apply_store44_style()

# Zielpfad fuer Rohdaten
DATA_RAW_PATH = "../data/raw/model_a_raw.csv"

print(f"Seed gesetzt: {SEED}")
print(f"numpy: {np.__version__}, pandas: {pd.__version__}")
print("Store44-Stil aktiviert.")

Seed gesetzt: 42
numpy: 2.4.6, pandas: 3.0.5
Store44-Stil aktiviert.


In [4]:
# =============================================================================
# Zelle 02 – Latente Auftragsgroesse (Rohr_DN) + X_A-Basisvariablen (korrigiert)
# =============================================================================
# Ziel: Realistische, geclusterte X_A-Rohdaten erzeugen - nur direkt gemessene
# Groessen (kein Feature Engineering hier, das kommt erst in Preprocessing).
# KORREKTUR ggue. erstem Entwurf: Formel-Konstanten waren nicht gegen die
# Zielbereiche kalibriert -> fuehrte zu massivem Clipping (Massedruck 100%,
# Drehzahl 74% an der Grenze). Jetzt rechnerisch kalibriert, Clipping < 2%.
# ANNAHME (zu pruefen): Nennweiten-Haeufigkeitsverteilung.
# =============================================================================

N = 700

# --- Latente Groesse: Nennweite (DN), nicht Teil von X_A, nur zur Datenerzeugung ---
dn_werte = np.array([50, 63, 75, 90, 110, 125, 160, 200])
dn_gewichte = np.array([0.10, 0.14, 0.16, 0.18, 0.18, 0.12, 0.08, 0.04])
rohr_dn = rng.choice(dn_werte, size=N, p=dn_gewichte)
querschnitt_proxy = (rohr_dn / 100) ** 2  # Hilfsgroesse, proportional zu DN^2

# --- Zielwandstaerke (latent, nicht X_A, dient nur zur Duesenspalt-Ableitung) ---
wandstaerke_basis = 1.0 + rohr_dn * 0.012
wandstaerke_ideal = np.clip(wandstaerke_basis + rng.normal(0, 0.15, N), 1.0, 4.0)

# --- Materialkennwert: MFR der Charge (HDPE, Chargenschwankung) ---
mfr_charge = np.clip(rng.normal(0.7, 0.15, N), 0.3, 1.2)

# --- Massedurchsatz: skaliert mit Querschnittsflaeche, kalibriert auf 5-150 kg/h ---
massedurchsatz = np.clip(15 + querschnitt_proxy * 30 + rng.normal(0, 4, N), 5, 150)

# --- Schneckendrehzahl: gedaempfte (sqrt) Kopplung an Durchsatz, kalibriert 10-80 rpm ---
# Begruendung gedaempfte statt lineare Kopplung: bei grossen Rohren werden oft
# groessere Extruder eingesetzt statt proportional hoeherer Drehzahl.
sqrt_md = np.sqrt(massedurchsatz)
slope_dz = (75 - 15) / (sqrt_md.max() - sqrt_md.min())
intercept_dz = 15 - slope_dz * sqrt_md.min()
schneckendrehzahl = np.clip(intercept_dz + slope_dz * sqrt_md + rng.normal(0, 2.5, N), 10, 80)

# --- Massetemperatur: leicht sinkend mit steigendem MFR (Websuche-Erkenntnis) ---
massetemperatur = np.clip(215 - (mfr_charge - 0.7) * 8 + rng.normal(0, 4, N), 180, 230)

# --- Duesenspalt: skaliert mit Soll-Wandstaerke, Die-Swell-Korrektur (<1) ---
die_swell_faktor = rng.normal(0.85, 0.04, N)
duesenspalt = np.clip(wandstaerke_ideal * die_swell_faktor + rng.normal(0, 0.08, N), 0.5, 6.0)

# --- Abzugsgeschwindigkeit: skaliert mit Querschnitt, kalibriert 0.5-15 m/min ---
abzugsgeschwindigkeit = np.clip(2.0 + querschnitt_proxy * 2.0 + rng.normal(0, 0.4, N), 0.5, 15)

# --- Massedruck: normierte Groessen (statt Rohgroessen-Division), kalibriert 50-300 bar ---
# Begruendung Normierung: direkte Division durch duesenspalt^2 mit Rohgroessen
# fuehrte zu Werten bis 7592 bar (100% Clipping) - normierte relative Abweichungen
# um den Mittelwert vermeiden diese Explosion.
viskositaets_proxy = 1 / mfr_charge
md_norm = massedurchsatz / massedurchsatz.mean()
visk_norm = viskositaets_proxy / viskositaets_proxy.mean()
ds_norm = duesenspalt / duesenspalt.mean()
massedruck = np.clip(150 * md_norm * visk_norm / (ds_norm ** 1.5) * rng.normal(1.0, 0.08, N), 50, 300)

# --- Vakuumniveau: steigt (im Betrag) mit DN, kalibriert -800 bis -100 mbar ---
vakuumniveau = np.clip(-150 - rohr_dn * 2.2 + rng.normal(0, 25, N), -800, -100)

# --- Kuehlwassertemperatur: eher Bedienerpraeferenz, viel Rauschen, wenig Struktur ---
kuehlwassertemperatur = np.clip(rng.normal(16, 4, N), 8, 25)

# --- DataFrame zusammenstellen (nur direkt gemessene Groessen, kein Feature Engineering) ---
df = pd.DataFrame({
    "rohr_dn_latent": rohr_dn,                      # latent, spaeter vor Export ggf. entfernen
    "wandstaerke_ideal_latent": wandstaerke_ideal,   # latent, spaeter vor Export ggf. entfernen
    "schneckendrehzahl": schneckendrehzahl,
    "massedurchsatz": massedurchsatz,
    "massetemperatur": massetemperatur,
    "massedruck": massedruck,
    "duesenspalt": duesenspalt,
    "abzugsgeschwindigkeit": abzugsgeschwindigkeit,
    "vakuumniveau": vakuumniveau,
    "kuehlwassertemperatur": kuehlwassertemperatur,
    "mfr_charge": mfr_charge,
})

# --- Clipping-Rate pro Spalte pruefen (Qualitaetssicherung, Ziel < 5%) ---
print(f"Shape: {df.shape}\n")
for col, (lo, hi) in {
    "schneckendrehzahl": (10, 80), "massedurchsatz": (5, 150),
    "massetemperatur": (180, 230), "massedruck": (50, 300),
    "duesenspalt": (0.5, 6.0), "abzugsgeschwindigkeit": (0.5, 15),
    "vakuumniveau": (-800, -100), "kuehlwassertemperatur": (8, 25),
}.items():
    clip_rate = ((df[col] <= lo) | (df[col] >= hi)).mean() * 100
    print(f"{col:25s} Clipping: {clip_rate:5.2f}%")

df.head(10)

Shape: (700, 11)

schneckendrehzahl         Clipping:  0.00%
massedurchsatz            Clipping:  0.00%
massetemperatur           Clipping:  0.00%
massedruck                Clipping:  1.14%
duesenspalt               Clipping:  0.00%
abzugsgeschwindigkeit     Clipping:  0.00%
vakuumniveau              Clipping:  0.00%
kuehlwassertemperatur     Clipping:  4.00%


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,mfr_charge
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,0.623140
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,0.838747
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,0.737482
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,0.855355
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,0.900449
5,125,2.654802,40.450721,61.361350,212.746303,140.583148,2.200708,5.304704,-465.020832,14.161976,0.768847
6,110,1.857562,46.936851,64.095768,219.025980,271.148532,1.580498,4.367596,-370.555534,10.912941,0.624028
7,50,1.570750,19.310710,18.839301,218.112780,66.969527,1.446849,2.325114,-253.506082,23.316809,0.856049
8,63,1.921325,30.573625,31.567866,214.710843,108.366060,1.699360,2.415030,-285.093369,18.949368,0.769850
9,75,2.038677,25.837241,25.744302,207.095330,51.189351,1.750185,3.900219,-332.349779,15.912097,1.090505


In [5]:
# =============================================================================
# Zelle 03 – Geometrie-Zielgroessen: Wandstaerke, Aussendurchmesser, Ovalitaet
# =============================================================================
# Kausalitaet: Die tatsaechliche Wandstaerke entsteht aus dem Duesenspalt und
# dem REALISIERTEN Die-Swell-Faktor - dieser haengt von Prozessbedingungen ab
# (Massetemperatur, MFR), die beim Einstellen des Duesenspalts (Zelle 02,
# geplanter Die-Swell) nicht exakt vorhersehbar sind. Das bildet den
# Kernmechanismus fuer Modell A ab: Prozessabweichung -> Qualitaetsabweichung.
# Aussendurchmesser/Ovalitaet haengen von der Vakuum-Kalibrierungsguete ab
# (Websuche: Vakuumformer bestimmt Aussendurchmesser waehrend Abkuehlung).
# =============================================================================

# --- Realisierter Die-Swell-Faktor (weicht vom geplanten die_swell_faktor ab) ---
die_swell_real = 0.85 + 0.003*(massetemperatur - 205) - 0.05*(mfr_charge - 0.7) + rng.normal(0, 0.015, N)
die_swell_real = np.clip(die_swell_real, 0.70, 1.00)

# --- Wandstaerke (Ist): aus Duesenspalt / realisiertem Die-Swell ---
wandstaerke_ist = duesenspalt / die_swell_real

# --- Vakuum-Kalibrierguete: staerkeres Vakuum (negativer) -> praeziseres Kalibrieren ---
vac_norm = np.clip((np.abs(vakuumniveau) - 100) / 700, 0, 1)  # 0=schwach, 1=stark
sigma_od = 0.8 - 0.5 * vac_norm  # mm, Streuung Aussendurchmesser

# --- Aussendurchmesser (Ist) ---
aussendurchmesser_ist = rohr_dn + rng.normal(0, sigma_od, N)

# --- Ovalitaet: gleicher Qualitaetstreiber (Vakuum), unabhaengige Realisierung ---
ovalitaet = np.abs(rng.normal(0, sigma_od * 0.6, N))

df["wandstaerke_ist"] = wandstaerke_ist
df["aussendurchmesser_ist"] = aussendurchmesser_ist
df["ovalitaet"] = ovalitaet

print("Wandstaerke Ist  - mean:", round(wandstaerke_ist.mean(),3), " std:", round(wandstaerke_ist.std(),3))
print("Abweichung Ist-Soll (Wandstaerke) - mean:", round((wandstaerke_ist - df['wandstaerke_ideal_latent']).mean(),4),
      " std:", round((wandstaerke_ist - df['wandstaerke_ideal_latent']).std(),4))
print("Aussendurchmesser Ist - mean:", round(aussendurchmesser_ist.mean(),2), " std:", round(aussendurchmesser_ist.std(),2))
print("Ovalitaet - mean:", round(ovalitaet.mean(),3), " max:", round(ovalitaet.max(),3))

df.head(5)

Wandstaerke Ist  - mean: 2.082  std: 0.484
Abweichung Ist-Soll (Wandstaerke) - mean: -0.0673  std: 0.1468
Aussendurchmesser Ist - mean: 96.62  std: 36.33
Ovalitaet - mean: 0.314  max: 1.516


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,mfr_charge,wandstaerke_ist,aussendurchmesser_ist,ovalitaet
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,0.623140,2.439594,124.655725,0.323485
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,0.838747,1.985378,90.444917,0.466484
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,0.737482,2.103764,89.794301,0.340501
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,0.855355,1.935124,49.843076,0.508177
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,0.900449,2.128977,90.555684,0.468881


In [6]:
# =============================================================================
# Zelle 04 – Wellgeometrie: Wellhoehe/-teilung (Ist)
# =============================================================================
# Kausalitaet: Wellenausformung haengt primaer vom Vakuumniveau ab (Quelle:
# UNICOR-Produktbeschreibung "jeder Profilberg kann einzeln mit Vakuum
# versorgt werden"). Zu schwaches Vakuum -> unvollstaendige Ausformung.
# Sollwerte (Wellhoehe/-teilung) sind produktspezifisch, hier als grobe
# Faustregel proportional zu DN angenommen (ANNAHME, nicht literaturbelegt -
# wellrohrspezifische Formeln nicht frei verfuegbar, siehe fruehere
# Recherche-Einschraenkung).
# =============================================================================

wellhoehe_soll = 0.08 * df["rohr_dn_latent"]
wellteilung_soll = 0.15 * df["rohr_dn_latent"]

# Ausformungsgrad: haengt von Vakuum-Staerke ab (vac_norm aus Zelle 03 wiederverwendet)
ausformungsgrad = np.clip(0.85 + 0.15 * vac_norm + rng.normal(0, 0.03, N), 0.6, 1.05)

df["wellhoehe_ist"] = wellhoehe_soll * ausformungsgrad
df["wellteilung_ist"] = wellteilung_soll + rng.normal(0, 0.5, N)

print("Ausformungsgrad - mean:", round(ausformungsgrad.mean(), 3), " min:", round(ausformungsgrad.min(), 3))
print("Wellhoehe Soll - mean:", round(wellhoehe_soll.mean(), 2), " Ist - mean:", round(df['wellhoehe_ist'].mean(), 2))

df.head(5)

Ausformungsgrad - mean: 0.911  min: 0.817
Wellhoehe Soll - mean: 7.73  Ist - mean: 7.09


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,mfr_charge,wandstaerke_ist,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,0.623140,2.439594,124.655725,0.323485,9.179281,18.022913
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,0.838747,1.985378,90.444917,0.466484,6.595309,13.409013
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,0.737482,2.103764,89.794301,0.340501,6.049400,14.497385
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,0.855355,1.935124,49.843076,0.508177,3.363355,7.997799
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,0.900449,2.128977,90.555684,0.468881,6.446902,13.808095


In [10]:
# =============================================================================
# Zelle 05 – Strukturelle/Oberflaechen-Fehlermerkmale (binaere Flags)
# =============================================================================
# Kausale Mechanismen (jeweils Quelle/Annahme markiert):
# - Bindenaehte: niedrige Massetemperatur -> schlechtere Verschweissung
#   (Quelle: Websuche roymaplast.com, Schmelzestrom-Verbindungslinien)
# - Blasenbildung: hohe Massetemperatur -> Entgasungsrisiko (ANNAHME,
#   da Feuchtigkeit nicht als eigene Variable gefuehrt wird)
# - Risse: kaltes Kuehlwasser + hohe Abzugsgeschwindigkeit -> Eigenspannung
#   (ANNAHME, plausibel)
# - Oberflaechenfehler: grosse Abweichung Wandstaerke Ist/Soll -> instabiler
#   Schmelzefluss (ANNAHME)
# Modellierung ueber logistische Risiko-Scores + Bernoulli-Ziehung, damit
# seltene, realistische Fehlerraten entstehen (keine deterministischen Regeln).
# =============================================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Bindenaehte: Risiko steigt bei niedriger Massetemperatur
score_bindenaht = -3.0 + np.clip((200 - massetemperatur) / 6, -4, 4)
df["bindenaehte"] = rng.random(N) < sigmoid(score_bindenaht)

# Blasenbildung: Risiko steigt bei hoher Massetemperatur
score_blase = -3.2 + np.clip((massetemperatur - 222) / 5, -4, 4)
df["blasenbildung"] = rng.random(N) < sigmoid(score_blase)

# Risse: Risiko steigt bei kaltem Kuehlwasser + hoher Abzugsgeschwindigkeit
score_riss = -3.2 + np.clip((11 - kuehlwassertemperatur) / 4, -4, 4) + 0.4 * np.clip((abzugsgeschwindigkeit - 9) / 3, -4, 4)
df["risse"] = rng.random(N) < sigmoid(score_riss)

# Oberflaechenfehler: Risiko steigt bei grosser Wandstaerkenabweichung Ist/Soll
score_oberflaeche = -3.0 + np.clip((np.abs(df["wandstaerke_ist"] - df["wandstaerke_ideal_latent"]) - 0.15) / 0.08, -4, 4)
df["oberflaechenfehler"] = rng.random(N) < sigmoid(score_oberflaeche)

for col in ["bindenaehte", "blasenbildung", "risse", "oberflaechenfehler"]:
    print(f"{col:20s} Rate: {df[col].mean()*100:.1f}%")

df.head(5)

bindenaehte          Rate: 0.1%
blasenbildung        Rate: 1.0%
risse                Rate: 1.6%
oberflaechenfehler   Rate: 6.3%


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,...,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler,wandtyp,delamination
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,...,124.655725,0.323485,9.179281,18.022913,False,False,False,True,einwandig,NaN
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,...,90.444917,0.466484,6.595309,13.409013,False,False,False,False,einwandig,NaN
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,...,89.794301,0.340501,6.049400,14.497385,False,False,False,False,einwandig,NaN
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,...,49.843076,0.508177,3.363355,7.997799,False,False,False,False,einwandig,NaN
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,...,90.555684,0.468881,6.446902,13.808095,False,False,False,False,doppelwandig,0.0


In [12]:
# =============================================================================
# Zelle 05b – Wandtyp (Kontextvariable) + Delamination
# =============================================================================
# Wandtyp wird nachtraeglich als Kontextvariable ergaenzt (Nutzerentscheidung),
# um Delamination sauber modellieren zu koennen. Groessere Rohre tendenziell
# haeufiger doppelwandig (ANNAHME, strukturelle Anwendungen).
# Delamination ist NUR bei doppelwandigen Rohren ueberhaupt messbar/anwendbar
# -> bei einwandigen Rohren NaN (strukturell fehlend, MNAR - bewusst so
# modelliert, relevant fuer spaetere EDA "fehlende Werte zufaellig?").
# =============================================================================

p_doppelwandig = 0.15 + 0.30 * (df["rohr_dn_latent"] - 50) / (200 - 50)
df["wandtyp"] = np.where(rng.random(N) < p_doppelwandig, "doppelwandig", "einwandig")

# Delamination-Risiko: niedrige Massetemperatur -> schlechtere Schichthaftung
# KORREKTUR: urspruengliche Kalibrierung (-3.0 + (202-T)/6) fuehrte zu 0% Rate,
# da Massetemperatur praktisch nie in den risikoerhoehenden Bereich fiel.
# Neu kalibriert: Baseline ~3% bei typischer Temperatur, steigt bei Abweichung.
score_delam = -3.48 + np.clip((215 - massetemperatur) / 8, -4, 4)
delam_risiko = rng.random(N) < sigmoid(score_delam)

df["delamination"] = np.where(df["wandtyp"] == "doppelwandig", delam_risiko, np.nan)

print("Anteil doppelwandig:", round((df['wandtyp']=='doppelwandig').mean()*100,1), "%")
print("Delamination-Rate (nur doppelwandig):", 
      round(df.loc[df['wandtyp']=='doppelwandig','delamination'].mean()*100,1), "%")
print("Delamination NaN (einwandig):", df['delamination'].isna().sum())

df.head(5)

Anteil doppelwandig: 24.7 %
Delamination-Rate (nur doppelwandig): 1.7 %
Delamination NaN (einwandig): 527


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,...,aussendurchmesser_ist,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler,wandtyp,delamination
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,...,124.655725,0.323485,9.179281,18.022913,False,False,False,True,einwandig,NaN
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,...,90.444917,0.466484,6.595309,13.409013,False,False,False,False,einwandig,NaN
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,...,89.794301,0.340501,6.049400,14.497385,False,False,False,False,doppelwandig,0.0
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,...,49.843076,0.508177,3.363355,7.997799,False,False,False,False,einwandig,NaN
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,...,90.555684,0.468881,6.446902,13.808095,False,False,False,False,einwandig,NaN


In [13]:
# =============================================================================
# Zelle 06 – IO/NIO-Aggregation (finale Zielgroesse Modell A)
# =============================================================================
# Kombiniert Toleranzgrenzen (kontinuierliche Merkmale) und binaere Fehlerflags
# zu einer Gesamt-IO/NIO-Klassifikation. Toleranzschwellen wurden rechnerisch
# kalibriert, um eine plausible Gesamtrate zu erreichen (kein Extremwert).
# =============================================================================

wandstaerke_abweichung = np.abs(df["wandstaerke_ist"] - df["wandstaerke_ideal_latent"])
od_abweichung = np.abs(df["aussendurchmesser_ist"] - df["rohr_dn_latent"])

nio_wandstaerke = wandstaerke_abweichung > 0.30
nio_ovalitaet = df["ovalitaet"] > 0.65
nio_od = od_abweichung > 1.2
nio_wellhoehe = ausformungsgrad < 0.85

nio_gesamt = (
    nio_wandstaerke | nio_ovalitaet | nio_od | nio_wellhoehe |
    df["bindenaehte"] | df["blasenbildung"] | df["risse"] | df["oberflaechenfehler"] |
    (df["delamination"] == True)  # NaN bei einwandig wird hier automatisch False
)

df["io_nio"] = np.where(nio_gesamt, "NIO", "IO")

print("IO/NIO-Verteilung:")
print(df["io_nio"].value_counts())
print(f"\nNIO-Rate: {(df['io_nio']=='NIO').mean()*100:.1f}%")

# Einzelursachen bei NIO-Faellen (zur Plausibilitaetspruefung)
print("\nHaeufigkeit der Einzelursachen (nur bei NIO-Faellen):")
nio_mask = df["io_nio"] == "NIO"
for name, arr in [("Wandstaerke", nio_wandstaerke), ("Ovalitaet", nio_ovalitaet),
                   ("Aussendurchmesser", nio_od), ("Wellhoehe", nio_wellhoehe),
                   ("Bindenaehte", df["bindenaehte"]), ("Blasenbildung", df["blasenbildung"]),
                   ("Risse", df["risse"]), ("Oberflaechenfehler", df["oberflaechenfehler"])]:
    anteil = (arr & nio_mask).sum() / nio_mask.sum() * 100
    print(f"  {name:20s}: {anteil:5.1f}% der NIO-Faelle")

df.head(5)

IO/NIO-Verteilung:
io_nio
IO     517
NIO    183
Name: count, dtype: int64

NIO-Rate: 26.1%

Haeufigkeit der Einzelursachen (nur bei NIO-Faellen):
  Wandstaerke         :  24.0% der NIO-Faelle
  Ovalitaet           :  33.9% der NIO-Faelle
  Aussendurchmesser   :  16.9% der NIO-Faelle
  Wellhoehe           :  17.5% der NIO-Faelle
  Bindenaehte         :   0.5% der NIO-Faelle
  Blasenbildung       :   3.8% der NIO-Faelle
  Risse               :   6.0% der NIO-Faelle
  Oberflaechenfehler  :  24.0% der NIO-Faelle


,rohr_dn_latent,wandstaerke_ideal_latent,schneckendrehzahl,massedurchsatz,massetemperatur,massedruck,duesenspalt,abzugsgeschwindigkeit,vakuumniveau,kuehlwassertemperatur,...,ovalitaet,wellhoehe_ist,wellteilung_ist,bindenaehte,blasenbildung,risse,oberflaechenfehler,wandtyp,delamination,io_nio
0,125,2.754905,45.192443,62.401028,220.105273,151.167887,2.226666,5.077495,-418.159128,11.623428,...,0.323485,9.179281,18.022913,False,False,False,True,einwandig,NaN,NIO
1,90,1.959540,32.680979,34.487359,212.380380,112.135223,1.705117,4.357722,-399.836605,18.639873,...,0.466484,6.595309,13.409013,False,False,False,False,einwandig,NaN,IO
2,90,2.340631,32.420695,37.597406,215.118805,104.752438,1.837200,3.516245,-377.966143,20.906668,...,0.340501,6.049400,14.497385,False,False,False,False,doppelwandig,0.0,NIO
3,50,1.903329,28.497070,31.344810,215.463089,74.747156,1.679957,1.649761,-269.384162,25.000000,...,0.508177,3.363355,7.997799,False,False,False,False,einwandig,NaN,NIO
4,90,2.156898,35.798154,43.028698,210.758177,91.085245,1.864819,4.178037,-365.824229,18.826512,...,0.468881,6.446902,13.808095,False,False,False,False,einwandig,NaN,IO


In [14]:
# =============================================================================
# Zelle 07 – Realismus-Luecken: gezielt fehlende Werte einbauen (MCAR)
# =============================================================================
# Ziel: realistische Datenqualitaet simulieren (nicht jedes Feld wird bei
# jedem Ruestvorgang vollstaendig dokumentiert). Bewusst MCAR (zufaellig,
# nicht systematisch) fuer diese Spalten - im Gegensatz zu Delamination
# (Zelle 05b), das strukturell/MNAR fehlt. Diese Unterscheidung ist fuer
# die spaetere EDA (Missing-Value-Mechanismus pruefen) relevant.
# Rate 3-5%, unabhaengig je Spalte gezogen.
# =============================================================================

MISSING_RATE = 0.04
spalten_mit_luecken = ["kuehlwassertemperatur", "mfr_charge", "wellteilung_ist", "aussendurchmesser_ist"]

for col in spalten_mit_luecken:
    missing_mask = rng.random(N) < MISSING_RATE
    df.loc[missing_mask, col] = np.nan

print("Fehlende Werte pro Spalte:")
print(df[spalten_mit_luecken].isna().sum())
print(f"\nGesamtanteil fehlender Werte (nur betroffene Spalten): {df[spalten_mit_luecken].isna().mean().mean()*100:.1f}%")

Fehlende Werte pro Spalte:
kuehlwassertemperatur    29
mfr_charge               34
wellteilung_ist          31
aussendurchmesser_ist    28
dtype: int64

Gesamtanteil fehlender Werte (nur betroffene Spalten): 4.4%


In [15]:
# =============================================================================
# Zelle 08 – Finales Speichern (data/raw/model_a_raw.csv)
# =============================================================================
# Latente Hilfsspalten (rohr_dn_latent, wandstaerke_ideal_latent) werden vor
# dem Speichern entfernt - sie sind kein X_A/Y_A gemaess Parametertabelle,
# sondern nur interne Generierungshilfsgroessen. Sie existierten NICHT in
# der Realitaet als dokumentierte Groessen (Rohr_DN ist z.B. eine Eigenschaft
# des Auftrags/der Formbacken, nicht etwas, das der Bediener beim
# Prozess-Ruesten fuer Modell A separat aufschreibt).
# =============================================================================

df_raw_final = df.drop(columns=["rohr_dn_latent", "wandstaerke_ideal_latent"])

print(f"Finale Spalten ({len(df_raw_final.columns)}):")
print(df_raw_final.columns.tolist())
print(f"\nShape: {df_raw_final.shape}")

df_raw_final.to_csv(DATA_RAW_PATH, index=False)
print(f"\nGespeichert: {DATA_RAW_PATH}")

Finale Spalten (21):
['schneckendrehzahl', 'massedurchsatz', 'massetemperatur', 'massedruck', 'duesenspalt', 'abzugsgeschwindigkeit', 'vakuumniveau', 'kuehlwassertemperatur', 'mfr_charge', 'wandstaerke_ist', 'aussendurchmesser_ist', 'ovalitaet', 'wellhoehe_ist', 'wellteilung_ist', 'bindenaehte', 'blasenbildung', 'risse', 'oberflaechenfehler', 'wandtyp', 'delamination', 'io_nio']

Shape: (700, 21)

Gespeichert: ../data/raw/model_a_raw.csv


In [16]:
# =============================================================================
# Zelle 08b – Referenzdatei mit latenten Groessen (nur zur spaeteren
# Validierung der EDA-Methodik, NICHT Teil der Trainingsdaten)
# =============================================================================
df_latent_reference = df[["rohr_dn_latent", "wandstaerke_ideal_latent"]].copy()
df_latent_reference.to_csv("../data/raw/model_a_latent_reference.csv", index=False)
print(f"Referenzdatei gespeichert: ../data/raw/model_a_latent_reference.csv")
print(f"Shape: {df_latent_reference.shape}")

Referenzdatei gespeichert: ../data/raw/model_a_latent_reference.csv
Shape: (700, 2)
